In [ ]:
# ============================================
# CELL 1: Install dependencies (MUST run first)
# ============================================
# Pin numpy<2 BEFORE anything else to avoid binary incompatibility
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy<2", "scipy<1.17", "scikit-learn", "timm", "facenet-pytorch"],
               check=True)
print("Dependencies installed. If you see numpy warnings above, they are safe to ignore.")

In [ ]:
# ============================================
# CELL 2: Clone repo, find dataset, run training
# ============================================
import os, sys, subprocess, time, json

# --- Clone project ---
if not os.path.exists('/kaggle/working/project'):
    subprocess.run(["git", "clone", "https://github.com/Jokeera/deepfake-detection.git",
                     "/kaggle/working/project"], check=True)
os.chdir('/kaggle/working/project')
sys.path.insert(0, '/kaggle/working/project')

import numpy as np
print(f"numpy version: {np.__version__}")
assert int(np.__version__.split('.')[0]) < 2, f"numpy 2.x detected ({np.__version__})! Restart kernel and re-run."

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU!"}')
assert torch.cuda.is_available(), "No GPU detected! Enable GPU in Kaggle settings."

# --- Find dataset ---
print("\n=== Searching for dataset in /kaggle/input/ ===")
data_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'real' in dirs and 'fake' in dirs:
        real_p = os.path.join(root, 'real')
        fake_p = os.path.join(root, 'fake')
        # Verify they actually contain data (subdirectories = video IDs)
        rc = len([d for d in os.listdir(real_p) if os.path.isdir(os.path.join(real_p, d))])
        fc = len([d for d in os.listdir(fake_p) if os.path.isdir(os.path.join(fake_p, d))])
        if rc > 0 and fc > 0:
            data_path = root
            break

if data_path is None:
    print("ERROR: Dataset not found! Directory structure:")
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        if level < 4:
            indent = '  ' * level
            print(f'{indent}{os.path.basename(root)}/ ({len(dirs)} dirs, {len(files)} files)')
    raise FileNotFoundError('Cannot find real/fake dirs with video subdirectories in /kaggle/input/')

real_count = len([d for d in os.listdir(os.path.join(data_path, 'real'))
                  if os.path.isdir(os.path.join(data_path, 'real', d))])
fake_count = len([d for d in os.listdir(os.path.join(data_path, 'fake'))
                  if os.path.isdir(os.path.join(data_path, 'fake', d))])
print(f'Dataset: {data_path}')
print(f'Real: {real_count}, Fake: {fake_count}, Total: {real_count + fake_count}')

# --- Run experiments ---
from config import Config
from train import train
from utils import save_metrics

EXPERIMENTS = [
    {'name': 'A1_full_model',    'model_type': 'full',       'fusion_type': 'adaptive'},
    {'name': 'A2_spatial_only',  'model_type': 'spatial',    'fusion_type': 'adaptive'},
    {'name': 'A3_temporal_only', 'model_type': 'temporal',   'fusion_type': 'adaptive'},
    {'name': 'A4_sequential',   'model_type': 'sequential',  'fusion_type': 'adaptive'},
]

all_results = []
for i, exp in enumerate(EXPERIMENTS, 1):
    print(f"\n{'='*70}")
    print(f"[{i}/4] {exp['name']} (model_type={exp['model_type']})")
    print(f"{'='*70}\n")

    cfg = Config()
    cfg.dataset_root = data_path
    cfg.model_type = exp['model_type']
    cfg.fusion_type = exp['fusion_type']

    # Kaggle GPU optimized settings
    cfg.max_epochs = 30
    cfg.batch_size = 16
    cfg.patience = 7
    cfg.device = 'auto'
    cfg.use_amp = True
    cfg.num_workers = 2
    cfg.pin_memory = True

    # Use fixed split from repo (splits/split_seed42.json is in git)
    cfg.split_mode = 'fixed'
    cfg.splits_dir = './splits'
    cfg.output_dir = './experiments'

    t0 = time.time()
    try:
        metrics = train(cfg)
        metrics['experiment_name'] = exp['name']
        metrics['status'] = 'success'
        metrics['duration_min'] = round((time.time() - t0) / 60, 1)
        all_results.append(metrics)
        test = metrics.get('test', {})
        print(f"\n[OK] {exp['name']} done in {metrics['duration_min']} min")
        print(f"     AUC={test.get('auc',0):.4f}  Acc={test.get('accuracy',0):.4f}  F1={test.get('f1',0):.4f}")
    except Exception as e:
        elapsed = round((time.time() - t0) / 60, 1)
        print(f"\n[FAIL] {exp['name']} after {elapsed} min: {e}")
        import traceback; traceback.print_exc()
        all_results.append({
            'experiment_name': exp['name'],
            'status': 'failed',
            'error': str(e),
            'duration_min': elapsed,
        })

    # Save after each experiment (in case kernel dies)
    save_metrics(all_results, './experiments/all_results.json')

# --- Summary ---
print(f"\n{'='*70}")
print("RESULTS SUMMARY")
print(f"{'='*70}")
print(f"{'Experiment':<25} {'AUC':>8} {'Acc':>8} {'F1':>8} {'Time':>8}")
print('-' * 60)
for r in all_results:
    if r.get('status') == 'success':
        t = r.get('test', {})
        print(f"{r['experiment_name']:<25} {t.get('auc',0):>8.4f} {t.get('accuracy',0):>8.4f} {t.get('f1',0):>8.4f} {r.get('duration_min',0):>7.1f}m")
    else:
        print(f"{r['experiment_name']:<25} {'FAILED':>8} {r.get('error','')[:40]}")

# Pack results for download
os.system("tar czf /kaggle/working/results.tar.gz experiments/ 2>/dev/null")
print(f"\nresults.tar.gz ready for download")
os.system("ls -lh /kaggle/working/results.tar.gz")